# Chemistry modeling

In chemistry, theoretical models are used to rationalize and better understand a wide range of phenomena. We will put a few selected methods in context via application examples and individual lab work to show how they can be used to address concrete chemical questions.

Chemistry modeling is conducted with the use of software programs. In this course, we will apply the Python-driven software package [VeloxChem](https://veloxchem.org/docs/intro.html) {cite}`veloxchem`.

(sec:qc)= 
## Quantum chemistry

Quantum chemistry solves the equations of quantum mechanics to model molecular systems, or more specifically, the electrons in the system. The nuclei, on the other hand, are treated as point charges creating an attractive potential for the electrons.

The differential equation of quantum mechanics are turned into matrix equations suitable for operations on computers by introducing a basis set. Typically one adopts localized atomic orbitals for this purpose that are predefined and tabulated. Very many basis sets are available to serve as computational cost efficient alternatives for different modeling purposes. 

### Simulating IR spectra

The workflow for simulating the IR spectrum of the water molecule is presented below, inspired by the example in [eChem](https://kthpanor.github.io/echem/docs/tutorials/vib_ir_calc.html). The workflow consists of the following steps:

1. Import the VeloxChem module
2. Define a molecule and choose basis set
3. Optimize the electronic structure for the starting geometry
4. Optimize the molecular structure
5. Run a vibrational analysis to get normal modes and IR intensities
6. Plot the IR spectrum

In [ ]:
import veloxchem as vlx

In [ ]:
molecule = vlx.Molecule.read_name("water")
basis = vlx.MolecularBasis.read(molecule, "def2-svp")

In [ ]:
scf_drv = vlx.ScfRestrictedDriver()

scf_drv.xcfun = "blyp"
scf_drv.ri_coulomb = True

scf_results = scf_drv.compute(molecule, basis)

In [ ]:
opt_drv = vlx.OptimizationDriver(scf_drv)

opt_results = opt_drv.compute(molecule, basis, scf_results)

In [ ]:
opt_molecule = vlx.Molecule.read_xyz_string(opt_results['final_geometry'])

In [ ]:
vib_analysis = vlx.VibrationalAnalysis(scf_drv)

vib_analysis_results = vib_analysis.compute(opt_molecule, basis)

In [ ]:
vib_analysis.plot(vib_results=vib_analysis_results)

Below follows an excerpt from the program output presenting data for the three vibrational modes of water (angle bend, symmetric stretch, and asymmetric stretch):

```
                                                  Vibrational Analysis                                                   
                                                  ======================                                                  
                                                                                                                          
                                   Vibrational Mode      1                                                                
                                   ----------------------------------------------------                                   
                                   Harmonic frequency:                1612.83  cm**-1                                     
                                   Reduced mass:                       1.0819  amu                                        
                                   Force constant:                     1.6581  mdyne/A                                    
                                   IR intensity:                      45.2095  km/mol                                     
                                   Normal mode:                                                                           
                                                              X           Y           Z                                   
                                   1       O            -0.0089     -0.0402      0.0569                                   
                                   2       H            -0.3204      0.2034     -0.5946                                   
                                   3       H             0.4623      0.4347     -0.3079                                   
                                                                                                                          
                                                                                                                          
                                   Vibrational Mode      2                                                                
                                   ----------------------------------------------------                                   
                                   Harmonic frequency:                3632.29  cm**-1                                     
                                   Reduced mass:                       1.0462  amu                                        
                                   Force constant:                     8.1326  mdyne/A                                    
                                   IR intensity:                       0.6656  km/mol                                     
                                   Normal mode:                                                                           
                                                              X           Y           Z                                   
                                   1       O             0.0064      0.0289     -0.0409                                   
                                   2       H            -0.5769     -0.3848      0.1318                                   
                                   3       H             0.4752     -0.0741      0.5175                                   
                                                                                                                          
                                                                                                                          
                                   Vibrational Mode      3                                                                
                                   ----------------------------------------------------                                   
                                   Harmonic frequency:                3726.63  cm**-1                                     
                                   Reduced mass:                       1.0796  amu                                        
                                   Force constant:                     8.8338  mdyne/A                                    
                                   IR intensity:                      12.2127  km/mol                                     
                                   Normal mode:                                                                           
                                                              X           Y           Z                                   
                                   1       O            -0.0625     -0.0185     -0.0229                                   
                                   2       H             0.5531      0.4008     -0.1774                                   
                                   3       H             0.4397     -0.1072      0.5407                                   
```

(sec:md)=
## Molecular dynamics

A fully quantum mechanical treatment is very costly, and thus only applicable for relatively small systems. This means that unless new approximations are introduced, many systems that are of interest will be far beyond the capabilities of today’s computers. A small piece of protein from the human body can consist of several thousand atoms, and even very small systems can increase by hundreds of atoms when a solvent is added.

The simulation of large proteins and long time-scales can instead be consider using molecular dynamics (MD), in which we use simple Newtonian physics on entities (e.g. atoms) constructed to emulate the behavior of the molecular system. In this chapter we will introduce the basics for running an MD simulation with [openMM](https://openmm.org/).

### Modeling room temperature dynamics

OpenMM offers a high-performance toolkit for molecular simulations. It consists of a library of simulation features and an application layer, allowing end-users to use the program directly. The application layer allows openMM to be run as a stand-alone program, which is what we exploit in this course. 

### Running a basic MD simulation with openMM
The following example is taken from the [openMM user guide](http://docs.openmm.org/latest/userguide/application/02_running_sims.html#a-first-example).

Roughly, the script below carries out five actions:
1. loads the PDB file `input.pdb` defining a biomolecular system
2. parameterizes the system using the Amber14 force field and TIP3P-FB water model
3. minimizes the energy
4. runs a simulation of 10,000 steps with a Langevin integrator
5. saves a snapshot frame to a PDB file called output.pdb every 1000 time steps.